# tp-05a ベースラインと目的関数の比較（Phase 0・1）

**実行場所: ホスト PC（stg）。** 先に tp-01〜tp-04 を通すこと。

## 目的
LightGBM（固定パラメータ）で、**勝ち馬確率 p_win を作る目的関数**を比べ、基準（A1: 二値）に対する改善を測る。
モデリング戦略書 §3・§4 の A1〜A4・C1 と、前段予測（tp-04）の寄与を、同じ条件で評価する。

| ID | 方式 | p_win の作り方 |
|---|---|---|
| A1 | 二値（p_win・p_top2・p_top3 を別々に学習） | レース内で Σp=1（top2 は 2、top3 は 3）に反復比例で正規化（M1） |
| A1+S | A1 + tp-04 の前段 3 列（`s_time_pred` ほか） | 同上 |
| A3 | レース内 softmax（LightGBM のカスタム目的関数） | softmax(score) |
| A4a / A4b | lambdarank / rank_xendcg | softmax(score)。温度 T=1 と、検証 fold で最適化した T（楽観的な参考値） |
| C1 | A1 + レース内の相対化（z・順位・最大との差）を主要列に追加 | 同上 |
| A3+C1 | softmax + 相対化 | softmax(score) |
| C1+S / A3+C1+S | 上記に前段 3 列も加えた組み合わせ | それぞれ正規化 / softmax |

## 条件
- 特徴量: tp-03 の採用セット（B0′ + 履歴特徴量 127 列、`speed_*` なし）。2026 年は使わない。test 2025 は tp-06 まで使わない。
- CV: valid 2022・2023・2024 の expanding window（purge 7 日 + embargo 3 日）、3 seed、300 ラウンド固定。
- 採否は **スクリーニング = fold 2022・2023、確認 = fold 2024**。主指標は p_win の LogLoss（正規化後）、併記: race_nll、Brier、ECE、1 位的中率、NDCG@3。差はレース単位の paired bootstrap の 95% 信頼区間で見る。
- 実験は MLflow（`keiba-tp-model-screen`）に記録する。ホストの 6 コアは他ジョブ用に空け、メモリ使用量を各段階で記録する。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.project_env import load_project_dotenv

load_project_dotenv()
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import json, subprocess, time, gc, hashlib
import numpy as np
import pandas as pd
import mlflow
import psutil

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ablation as ab
from src.pipeline.models import win_models as wm
from src.pipeline.models.compute import detect_compute
from src.pipeline.models.time_split import yearly_expanding_folds

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OUT = WORK / "model_screen"
OUT.mkdir(parents=True, exist_ok=True)
COMPUTE = detect_compute(reserve_cores=6)
THREADS = COMPUTE.threads
SEEDS = [0, 1, 2]
MLFLOW_URI = require_tracking_server()
mlflow.set_experiment("keiba-tp-model-screen")
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
PROC = psutil.Process()


def mem(label=""):
    vm = psutil.virtual_memory()
    s = f"[メモリ] {label}: この process {PROC.memory_info().rss / 1e9:.1f}GB / ホスト使用 {vm.used / 1e9:.0f}GB ({vm.percent:.0f}%) / 空き {vm.available / 1e9:.0f}GB"
    print(s)
    return PROC.memory_info().rss / 1e9, vm.percent


print("MLflow:", MLFLOW_URI, "| git:", GIT_COMMIT, "| threads:", THREADS, "| seeds:", SEEDS)
mem("開始")

## 0. データ

In [ ]:
adopted = json.loads((WORK / "ablation_nospeed/adopted_features.json").read_text(encoding="utf-8"))
BASE = list(adopted["b0_features"]) + list(adopted["features"])
stage = pd.read_parquet(WORK / "oof/stage_features_expanding.parquet")
STAGE_COLS = ["s_time_pred", "s_time_gap", "s_rank_pred"]
df = ab.load_table(WORK / "features/train_table.parquet")
df["race_id"] = df["race_id"].astype(str)
df["horse_id"] = df["horse_id"].astype(str)
df = df.merge(stage[["race_id", "horse_id"] + STAGE_COLS], on=["race_id", "horse_id"], how="left")
# レース単位に連続して並べる（ランキング目的関数の group 用）
df = df.sort_values(["date", "race_id", "horse_number"], kind="stable").reset_index(drop=True)
RELCOLS = [c for c in ["f3_r", "sr_perf_ewm2", "sr_perf_ewm6", "sr_exp", "c_res_ewm2", "c_cls_ewm2", "a_l3fz_ewm2", "a_rk_ewm2", "a_style", "b_aff_all",
                       "pc_adv", "f2_pace_match", "horse_prior_win_rate", "jockey_prior_win_rate"] if c in df.columns]
REL = wm.add_race_relative(df, RELCOLS)
for c in REL.columns:
    df[c] = REL[c].to_numpy()
miss = [c for c in BASE + STAGE_COLS if c not in df.columns]
assert not miss, miss
assert int(df["year"].max()) == 2025
folds = list(yearly_expanding_folds(df))
print(f"行 {len(df):,} / レース {df['race_id'].nunique():,} / BASE {len(BASE)} 列 / 相対化 {len(REL.columns)} 列 / fold:", [(f.fold_id, len(f.train_index), len(f.valid_index)) for f in folds])
print("前段 3 列の非欠損率（年別）:", df.groupby("year")[STAGE_COLS[0]].apply(lambda s: round(float(s.notna().mean()), 3)).to_dict())
mem("データ読み込み後")

## 1. 学習・評価の共通部と実験の定義

In [ ]:
RACE = df["race_id"].to_numpy()
FIN = df["finish_position"].to_numpy()
Y = {"win": (FIN == 1).astype(int), "top2": (FIN <= 2).astype(int), "top3": (FIN <= 3).astype(int), "finish": FIN}
FEATS = {"base": BASE, "base+S": BASE + STAGE_COLS, "base+C1": BASE + list(REL.columns), "base+S+C1": BASE + STAGE_COLS + list(REL.columns)}
METHODS = {  # 名前: (方式, 特徴量セット)
    "A1_binary": ("binary", "base"),
    "A1_binary+S": ("binary", "base+S"),
    "A3_softmax": ("softmax", "base"),
    "A4a_lambdarank": ("lambdarank", "base"),
    "A4b_xendcg": ("xendcg", "base"),
    "C1_binary+rel": ("binary", "base+C1"),
    "A3_softmax+rel": ("softmax", "base+C1"),
    "C1_binary+rel+S": ("binary", "base+S+C1"),
    "A3_softmax+rel+S": ("softmax", "base+S+C1"),
}
PRED = {}    # 名前 -> {'win': p_win(全 valid 行, seed 平均), ...}
ROWS = []    # fold × seed ごとの指標


def best_temperature(race_ids, score, y_win):
    grid = np.exp(np.linspace(np.log(0.2), np.log(5.0), 25))
    return float(min(grid, key=lambda t: wm.race_nll(race_ids, wm.softmax_by_race(race_ids, score, t), y_win)))


def run_method(name):
    kind, fs = METHODS[name]
    X = df[FEATS[fs]]
    t0 = time.time()
    acc = {k: np.full(len(df), np.nan) for k in ("win", "top2", "top3", "win_T")}
    n_seed = np.zeros(len(df))
    for f in folds:
        tr, va = f.train_index, f.valid_index
        ytr = {k: v[tr] for k, v in Y.items()}
        per_seed = []
        for seed in SEEDS:
            out = wm.fit_predict(kind, X.iloc[tr], ytr, RACE[tr], X.iloc[va], RACE[va], THREADS, seed=seed)
            rv, yv = RACE[va], Y["win"][va].astype(float)
            if kind == "binary":
                raw_win = out["win"]
                p1 = wm.normalize_to_total(rv, out["win"], 1.0)
                p2 = wm.normalize_to_total(rv, out["top2"], 2.0)
                p3 = wm.normalize_to_total(rv, out["top3"], 3.0)
                m = wm.evaluate_probs(rv, p1, p2, p3, FIN[va])
                m["logloss_win_raw"] = wm.binary_logloss(raw_win, yv)    # 正規化前
                acc["win"][va] = np.nan_to_num(acc["win"][va]) + p1
                acc["top2"][va] = np.nan_to_num(acc["top2"][va]) + p2
                acc["top3"][va] = np.nan_to_num(acc["top3"][va]) + p3
            else:
                sc = out["win_score"]
                p1 = wm.softmax_by_race(rv, sc, 1.0)
                m = wm.evaluate_probs(rv, p1, None, None, FIN[va])
                if kind in ("lambdarank", "xendcg"):
                    T = best_temperature(rv, sc, yv)
                    pT = wm.softmax_by_race(rv, sc, T)
                    m["T_oracle"] = T
                    m["logloss_win_T"] = wm.binary_logloss(pT, yv)
                    m["race_nll_T"] = wm.race_nll(rv, pT, yv)
                    acc["win_T"][va] = np.nan_to_num(acc["win_T"][va]) + pT
                acc["win"][va] = np.nan_to_num(acc["win"][va]) + p1
            ROWS.append({"method": name, "fold": f"valid_{int(df['year'].iloc[va[0]])}", "seed": seed, "n_train": len(tr), **m})
        n_seed[va] = len(SEEDS)
    PRED[name] = {k: (v / np.where(n_seed > 0, n_seed, np.nan)) for k, v in acc.items() if not np.all(np.isnan(v))}
    r = pd.DataFrame([x for x in ROWS if x["method"] == name])
    print(f"  {name:18s} p_win LogLoss {r['logloss_win'].mean():.5f}  race_nll {r['race_nll'].mean():.4f}  ({time.time() - t0:.0f}s)")
    return r

## 2. 学習・評価（MLflow に 1 実験 1 run で記録）
各方式を fold 3 つ × seed 3 つで評価する。ホストのメモリを超えないよう 1 方式ずつ順に実行し、各方式の後にメモリ使用量を記録する。

In [ ]:
RES_PATH = OUT / "fold_metrics.csv"
for name in METHODS:
    with mlflow.start_run(run_name=f"tp-05a-{name}", tags={"notebook": "tp-05a", "git_commit": GIT_COMMIT, "method": name, "kind": METHODS[name][0], "features": METHODS[name][1],
                                                           "cv": "expanding valid 2022-2024", "training_years": "2020-2025"}):
        r = run_method(name)
        rss, pct = mem(name)
        mlflow.log_params({"kind": METHODS[name][0], "feature_set": METHODS[name][1], "n_features": len(FEATS[METHODS[name][1]]), "n_rounds": wm.N_ROUNDS,
                           "seeds": str(SEEDS), "threads": THREADS, **{f"lgb_{k}": v for k, v in wm.BASE_PARAMS.items() if k in ("learning_rate", "num_leaves", "min_data_in_leaf")}})
        num = [c for c in r.columns if c not in ("method", "fold", "seed", "n_train")]
        mlflow.log_metrics({f"mean_{c}": float(r[c].mean()) for c in num})
        for fold, g in r.groupby("fold"):
            mlflow.log_metrics({f"{fold}_{c}": float(g[c].mean()) for c in ("logloss_win", "race_nll", "top1_hit")})
        mlflow.log_metrics({"rss_gb": rss, "host_mem_percent": pct})
    pd.DataFrame(ROWS).to_csv(RES_PATH, index=False)
    gc.collect()

## 3. 結果

In [ ]:
R = pd.DataFrame(ROWS)
cols = ["logloss_win", "race_nll", "brier_win", "ece_win", "top1_hit", "ndcg3", "logloss_top2", "logloss_top3", "logloss_win_raw", "logloss_win_T", "race_nll_T", "T_oracle"]
cols = [c for c in cols if c in R.columns]
summary = R.groupby("method")[cols].mean().loc[list(METHODS)]
display(summary.round(5))
per_fold = R.groupby(["method", "fold"])["logloss_win"].mean().unstack("fold").loc[list(METHODS)]
per_fold["スクリーニング(2022・2023)"] = per_fold[["valid_2022", "valid_2023"]].mean(axis=1)
per_fold["確認(2024)"] = per_fold["valid_2024"]
per_fold["平均"] = per_fold[["valid_2022", "valid_2023", "valid_2024"]].mean(axis=1)
print("p_win LogLoss（fold 別）"); display(per_fold.round(5))
summary.to_csv(OUT / "summary.csv"); per_fold.to_csv(OUT / "per_fold_logloss.csv")

In [ ]:
# A1（二値）を基準に、レース単位の paired bootstrap で差（負なら改善）の 95% 信頼区間を出す
va_all = np.concatenate([f.valid_index for f in folds])
base_loss = wm.per_race_loss(RACE[va_all], PRED["A1_binary"]["win"][va_all], Y["win"][va_all].astype(float))
rows = []
for name in METHODS:
    if name == "A1_binary":
        continue
    pw = PRED[name]["win"]
    ok = ~np.isnan(pw[va_all])
    loss = wm.per_race_loss(RACE[va_all][ok], pw[va_all][ok], Y["win"][va_all][ok].astype(float))
    b = wm.paired_bootstrap_diff(base_loss, loss, n_boot=1000)
    folds_better = int(sum(per_fold.loc[name, f] < per_fold.loc["A1_binary", f] for f in ("valid_2022", "valid_2023", "valid_2024")))
    rows.append({"method": name, "Δ LogLoss(平均)": b["diff"], "95%下限": b["lo"], "95%上限": b["hi"], "改善した fold 数": folds_better,
                 "採用基準(上限<0 かつ 2fold以上)": bool(b["hi"] < 0 and folds_better >= 2)})
vs = pd.DataFrame(rows).set_index("method")
display(vs.round(5)); vs.to_csv(OUT / "vs_A1.csv")

In [ ]:
mem("評価後")
with mlflow.start_run(run_name="tp-05a-summary", tags={"notebook": "tp-05a", "git_commit": GIT_COMMIT}):
    for name in METHODS:
        safe = name.replace("+", "_plus_")   # MLflow の指標名に + は使えない
        mlflow.log_metric(f"logloss_win__{safe}", float(summary.loc[name, "logloss_win"]))
        mlflow.log_metric(f"race_nll__{safe}", float(summary.loc[name, "race_nll"]))
    for c in ("summary.csv", "per_fold_logloss.csv", "vs_A1.csv", "fold_metrics.csv"):
        mlflow.log_artifact(str(OUT / c), "model_screen")
best = summary["logloss_win"].idxmin()
report = {"git_commit": GIT_COMMIT, "best_by_logloss": best, "summary": summary.round(6).reset_index().to_dict("records"),
          "per_fold": per_fold.round(6).reset_index().to_dict("records"), "vs_A1": vs.round(6).reset_index().to_dict("records"),
          "n_rows": int(len(df)), "n_base_features": len(BASE), "stage_cols": STAGE_COLS, "seeds": SEEDS}
(OUT / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("最良（p_win LogLoss）:", best)